In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.stage")

In [ ]:
## Geronimo!!

In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd
from delta.tables import DeltaTable

In [0]:
ticker_rows = spark.table("stocks.reference.tickers").select("ticker").collect()

records = []
for row in ticker_rows:
    try:
        articles = yf.Ticker(row.ticker).news
        for a in articles:
            content  = a.get("content", a)
            uid      = content.get("id") or a.get("id")
            title    = content.get("title")
            summary  = content.get("summary") or ""
            pub_date = content.get("pubDate") or content.get("displayTime")
            publisher = (content.get("provider") or {}).get("displayName") or content.get("publisher")
            link     = ((content.get("canonicalUrl") or {}).get("url")
                        or (content.get("clickThroughUrl") or {}).get("url")
                        or a.get("link"))
            if uid and title:
                records.append({
                    "uuid":         uid,
                    "ticker":       row.ticker,
                    "title":        title,
                    "summary":      summary,
                    "publisher":    publisher,
                    "published_at": pub_date,
                    "link":         link,
                })
    except Exception as e:
        print(f"  SKIP {row.ticker}: {e}")

print(f"Fetched {len(records)} articles for {len(ticker_rows)} tickers")

In [0]:
records

In [0]:
pdf = pd.DataFrame(records).dropna(subset=["uuid", "title"])
spark_df = spark.createDataFrame(pdf)

tbl = "stocks.stage.news_raw"
if not spark.catalog.tableExists(tbl):
    spark_df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    spark_df.alias("s"), "s.uuid = t.uuid"
).whenNotMatchedInsertAll().execute()

print(f"stocks.stage.news_raw: {spark.table(tbl).count()} rows")